In [1]:
import cv2
import numpy as np
import os

print("Everything is working")

Everything is working


In [2]:
import os

print(os.listdir("input_images"))

['round_01.jpeg', 'round_02.jpeg', 'round_03.jpeg', 'round_04.jpeg', 'round_05.jpeg', 'round_06.jpeg', 'round_07.jpeg', 'square_01.jpeg', 'square_02.jpeg', 'square_03.jpeg', 'square_04.jpeg', 'square_05.jpeg', 'square_06.jpeg', 'square_07.jpeg']


In [4]:
import cv2
import numpy as np
import os

input_folder = "input_images"
output_folder = "output_images"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):

    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(input_folder, filename)
    image = cv2.imread(image_path)

    if image is None:
        print("Could not read:", filename)
        continue

    image = cv2.resize(image, (600, 800))
    output = image.copy()

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    lower_biscuit = np.array([5, 25, 40])
    upper_biscuit = np.array([40, 255, 255])

    mask = cv2.inRange(hsv, lower_biscuit, upper_biscuit)

    kernel_small = np.ones((3, 3), np.uint8)
    kernel_large = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_small)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_large)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    biscuit_count = 0
    intact_count = 0
    broken_count = 0

    for contour in contours:

        area = cv2.contourArea(contour)

        if area < 800:
            continue

        biscuit_count += 1

        x, y, w, h = cv2.boundingRect(contour)

        perimeter = cv2.arcLength(contour, True)

        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)

        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)

        if hull_area == 0:
            continue

        solidity = area / hull_area

        missing_area = hull_area - area
        missing_ratio = missing_area / hull_area

        rotated_rect = cv2.minAreaRect(contour)
        box = cv2.boxPoints(rotated_rect)
        box = np.intp(box)

        rotated_w = rotated_rect[1][0]
        rotated_h = rotated_rect[1][1]

        if rotated_w == 0 or rotated_h == 0:
            continue

        rotated_area = rotated_w * rotated_h
        rotated_extent = area / rotated_area

        rotated_aspect_ratio = max(rotated_w, rotated_h) / min(rotated_w, rotated_h)

        approx = cv2.approxPolyDP(contour, 0.03 * perimeter, True)
        corners = len(approx)

        name = filename.lower()

        if name.startswith("round"):

            if (
                area > 12000 and
                circularity > 0.68 and
                solidity > 0.91 and
                missing_ratio < 0.12
            ):
                label = "Intact Biscuit"
                color = (0, 255, 0)
                intact_count += 1
            else:
                label = "Broken Biscuit"
                color = (0, 0, 255)
                broken_count += 1

            cv2.rectangle(output, (x, y), (x + w, y + h), color, 2)

        elif name.startswith("square"):

            if (
                area > 14000 and
                rotated_extent > 0.68 and
                solidity > 0.91 and
                rotated_aspect_ratio < 1.45 and
                missing_ratio < 0.12 and
                corners >= 4
            ):
                label = "Intact Biscuit"
                color = (0, 255, 0)
                intact_count += 1
            else:
                label = "Broken Biscuit"
                color = (0, 0, 255)
                broken_count += 1

            cv2.drawContours(output, [box], 0, color, 2)

        else:
            label = "Biscuit"
            color = (255, 0, 0)
            cv2.rectangle(output, (x, y), (x + w, y + h), color, 2)

        cv2.putText(
            output,
            label,
            (x, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2
        )

    output_path = os.path.join(output_folder, "result_" + filename)
    cv2.imwrite(output_path, output)

    print(
        filename,
        "processed | Total:",
        biscuit_count,
        "| Intact:",
        intact_count,
        "| Broken:",
        broken_count
    )

print("Finished processing all images.")

round_01.jpeg processed | Total: 8 | Intact: 3 | Broken: 5
round_02.jpeg processed | Total: 8 | Intact: 3 | Broken: 5
round_03.jpeg processed | Total: 8 | Intact: 3 | Broken: 5
round_04.jpeg processed | Total: 8 | Intact: 3 | Broken: 5
round_05.jpeg processed | Total: 8 | Intact: 3 | Broken: 5
round_06.jpeg processed | Total: 8 | Intact: 3 | Broken: 5
round_07.jpeg processed | Total: 8 | Intact: 3 | Broken: 5
square_01.jpeg processed | Total: 7 | Intact: 3 | Broken: 4
square_02.jpeg processed | Total: 7 | Intact: 3 | Broken: 4
square_03.jpeg processed | Total: 7 | Intact: 3 | Broken: 4
square_04.jpeg processed | Total: 7 | Intact: 3 | Broken: 4
square_05.jpeg processed | Total: 7 | Intact: 3 | Broken: 4
Finished processing all images.
